# Cross-Country Comparison and Vulnerability Ranking

Task 3 implementation using cleaned outputs from all five country notebooks.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from src.data_loader import load_all
from src.eda_utils import extreme_heat_days, consecutive_dry_days, kruskal_wallis_t2m, vulnerability_summary
from src.visualization import multi_country_temperature

data_dir = Path("data")
clean_paths = list(data_dir.glob("*_clean.csv"))
if len(clean_paths) == 5:
    df = pd.concat([pd.read_csv(p, parse_dates=["DATE"]) for p in clean_paths], ignore_index=True)
else:
    print("Missing cleaned files, loading raw datasets as fallback")
    df = load_all(data_dir)

multi_country_temperature(df)
display(df.groupby("Country")["T2M"].agg(["mean", "median", "std"]).round(3))
plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x="Country", y="PRECTOTCORR")
plt.title("Precipitation variability by country")
plt.show()
display(df.groupby("Country")["PRECTOTCORR"].agg(["mean", "median", "std"]).round(3))
heat = extreme_heat_days(df)
dry = consecutive_dry_days(df)
plt.figure(figsize=(12, 5))
sns.barplot(data=heat, x="Year", y="extreme_heat_days", hue="Country")
plt.title("Extreme heat days (>35C) per year")
plt.xticks(rotation=45)
plt.show()
plt.figure(figsize=(12, 5))
sns.barplot(data=dry, x="Year", y="max_dry_streak", hue="Country")
plt.title("Maximum consecutive dry days per year")
plt.xticks(rotation=45)
plt.show()
print("Kruskal-Wallis:", kruskal_wallis_t2m(df))
ranking = vulnerability_summary(df)
display(ranking)
print("COP32 framing bullets to include in final report:")
print("- Fastest warming country and trend implication")
print("- Most unstable precipitation profile and implications")
print("- Heat/drought stress comparison")
print("- Ethiopia relative profile vs neighbors")
print("- Priority finance case supported by data")


## Data Loading and Integration (All Five Countries)

This section explicitly loads all cleaned country datasets and concatenates them into one analysis-ready table.

- `data/ethiopia_clean.csv`
- `data/kenya_clean.csv`
- `data/sudan_clean.csv`
- `data/tanzania_clean.csv`
- `data/nigeria_clean.csv`

The combined dataset is the basis for all cross-country comparisons, extreme-event analysis, and vulnerability ranking.

In [ ]:
from pathlib import Path
import pandas as pd

COUNTRIES = ["ethiopia", "kenya", "sudan", "tanzania", "nigeria"]
data_dir = Path("data")

frames = []
missing = []
for country in COUNTRIES:
    path = data_dir / f"{country}_clean.csv"
    if path.exists():
        df_c = pd.read_csv(path, parse_dates=["DATE"])
        if "Country" not in df_c.columns:
            df_c["Country"] = country.capitalize()
        frames.append(df_c)
    else:
        missing.append(path.name)

if missing:
    raise FileNotFoundError(f"Missing cleaned files: {missing}. Run country EDA cleaning/export first.")

combined = pd.concat(frames, ignore_index=True)
combined["Year"] = combined["DATE"].dt.year
combined["Month"] = combined["DATE"].dt.month

print("Combined shape:", combined.shape)
print("Countries loaded:", sorted(combined["Country"].unique().tolist()))
combined[["Country", "DATE", "T2M", "PRECTOTCORR"]].head()

## Temperature Comparison Across Five Countries

This section provides:
- a multi-country monthly temperature trend chart,
- a summary statistics table (`mean`, `median`, `std`) for `T2M`.

Interpretation should focus on relative warming behavior, variability, and which countries exhibit stronger thermal stress signatures.

## Precipitation Comparison Across Five Countries

This section provides:
- side-by-side precipitation boxplots for `PRECTOTCORR`,
- a summary statistics table (`mean`, `median`, `std`) for precipitation.

Interpretation should highlight variability, tail behavior, and countries with unstable rainfall patterns relevant to drought/flood planning.

## Extreme Events and Vulnerability Ranking

This section compares two core stress indicators:
- annual extreme heat frequency (`T2M_MAX > 35°C`),
- annual maximum consecutive dry days (`PRECTOTCORR < 1 mm`).

It then builds a vulnerability ranking table combining heat intensity, dry-spell persistence, and precipitation instability.

## COP32-Framed Key Observations

- **Fastest warming signal:** Identify which country shows the steepest multi-year rise in monthly mean `T2M`, and explain implications for heat adaptation.
- **Most unstable precipitation profile:** Identify the country with the broadest precipitation spread and strongest tails, and discuss flood-drought volatility.
- **Extreme climate stress:** Compare heat-day counts and dry-streak duration to show where climate stress is most persistent.
- **Ethiopia in regional context:** Explain where Ethiopia sits relative to neighbors on warming, rainfall variability, and extreme events.
- **Priority climate finance case:** Recommend one country Ethiopia should prioritize in COP32 advocacy, supported by ranking evidence and extreme-event burden.

## Methodological Assumptions and Transparency Notes

- Cross-country comparison uses cleaned daily records produced by the same preprocessing policy across all five countries.
- Extreme heat is defined as `T2M_MAX > 35°C`; drought persistence is approximated with consecutive days where `PRECTOTCORR < 1 mm`.
- Vulnerability ranking is a composite heuristic (heat frequency, dry-streak persistence, precipitation variability), not a causal risk model.
- Statistical significance (Kruskal-Wallis) tests distribution differences, not policy impact causality.
- Results support COP32 framing as climate-stress evidence and should be complemented with socioeconomic impact indicators for finance prioritization.